In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import *
import pyspark.sql.functions as F
from pyspark.sql.utils import AnalysisException
from pyspark.sql.types import StructType

In [0]:
display( 
        spark.read.option("header", "true")
        .csv(f"/Volumes/catalog_southeastasia_mdm_pr/share_mdm_config/mdm_config_files/cdp_mdm_pr/touchpoint_retry/pr_touchpoint_data_loading_delta_config_20260618.csv")
)


In [0]:
# ==============================================
# 【新增】Schema 表结构对比函数：打印字段/类型差异
# ==============================================
def compare_schemas(source_df, target_df, source_name="Source表", target_name="Target表"):
    """
    对比两个DataFrame的Schema结构，输出差异：
    1. Source有、Target没有的字段
    2. Target有、Source没有的字段
    3. 字段名相同、数据类型不同的字段
    """
    # 提取字段名+类型 转字典
    source_schema = {f.name.upper(): str(f.dataType).upper() for f in source_df.schema.fields}
    target_schema = {f.name.upper(): str(f.dataType).upper() for f in target_df.schema.fields}
    
    source_fields = set(source_schema.keys())
    target_fields = set(target_schema.keys())
    
    # 1. 独有字段
    source_only = source_fields - target_fields
    target_only = target_fields - source_fields
    
    # 2. 同名字段类型不同
    type_mismatch = []
    common_fields = source_fields & target_fields
    for field in common_fields:
        s_type = source_schema[field]
        t_type = target_schema[field]
        if s_type != t_type:
            type_mismatch.append(f"{field} | Source类型:{s_type} | Target类型:{t_type}")
    
    # 格式化打印差异
    print("-" * 80)
    print(f"📊 【表结构差异对比】{source_name} VS {target_name}")
    print("-" * 80)
    print(f"🔹 {source_name} 独有字段 ({len(source_only)}个): {sorted(source_only) if source_only else '无'}")
    print(f"🔹 {target_name} 独有字段 ({len(target_only)}个): {sorted(target_only) if target_only else '无'}")
    print(f"🔹 字段类型不一致 ({len(type_mismatch)}个):")
    if type_mismatch:
        for idx, msg in enumerate(type_mismatch, 1):
            print(f"   {idx}. {msg}")
    else:
        print("   无")
    print("-" * 80)

In [0]:
def align_schema(source_df: DataFrame, target_df: DataFrame) -> DataFrame:
    """
    自动将 source_df 的字段类型转换为与 target_df 完全一致的类型
    字段名必须匹配，自动按 target_df 字段顺序重排，自动类型强转
    
    参数：
        source_df: 原始数据源 DataFrame
        target_df: 目标表结构 DataFrame（用于获取目标 schema）
    
    返回：
        类型对齐后的新 DataFrame
    """

    # source_fields = [f.upper() for f in source_df.columns]
    # source_schema = source_df.schema
    # # 获取目标表的所有字段名 + 类型
    # target_fields = [f.upper() for f in target_df.columns]
    # target_schema = target_df.schema


    # 提取字段名+类型 转字典
    source_schema = {f.name.upper(): f.dataType for f in source_df.schema.fields}
    source_fields = set(source_schema.keys())

    target_schema = {f.name.upper(): f.dataType for f in target_df.schema.fields}
    target_fields = set(target_schema.keys())


    # 只保留 source_df 和 target_df 共有的字段
    common_fields = [field for field in target_fields if field in source_fields]
    print(f"common_fields: {common_fields}")

    # 为每个共有字段执行【自动类型转换】，并按 target 顺序排列
    casted_cols = []

    for field in common_fields:
        s_type = source_schema[field]
        t_type = target_schema[field]

        if s_type != t_type:
            casted_cols.append(
                F.col(field).cast(t_type).alias(field)
                )
        else:
            casted_cols.append(F.col(field))

    # 生成最终对齐后的 DataFrame
    result_df = source_df.select(*casted_cols)
    result_df.printSchema()

    return result_df

In [0]:
def org_field(level, field):
    return F.expr(f"""
        element_at(
            filter(
                from_json(
                    get_json_object(slntp_payload, '$.TouchPoint.OrganizationHierarchyList.OrganizationHierarchy'),
                    'array<struct<Level:string,Code:string,Name:string,Description:string>>'
                ),
                x -> x.Level = '{level}'
            ),
            1
        ).{field}
    """)


def get_add_json_column(json_df):
    
    json_parse_df = json_df.select(

        F.col("SLNTP_ID").alias("tcpm_tcpt_id"), #原 tcpt_sbat_id

        # Header
        F.get_json_object("slntp_payload", "$.Header.@Action").alias("tcpm_action"),
        F.get_json_object("slntp_payload", "$.Header.DocumentTimestamp").alias("tcpm_DOCUMENTTIMESTAMP"),
        F.get_json_object("slntp_payload", "$.Header.DocumentUUID").alias("tcpm_DOCUMENTUUID"),

        # TouchPoint root
        F.get_json_object("slntp_payload", "$.TouchPoint.@RecordUUID").alias("tcpm_RECORDUUID"),

        F.get_json_object("slntp_payload", "$.TouchPoint.Attributes.RedirectTouchPointCode").alias("tcpm_RedirectTouchPointCode"),
        # # >>> 新增 Attributes 字段 <<<
        F.get_json_object("slntp_payload", "$.TouchPoint.Attributes.Channel").alias("tcpm_Channel"),
        F.get_json_object("slntp_payload", "$.TouchPoint.Attributes.CustomerGroup").alias("tcpm_CustomerGroup"),
        F.get_json_object("slntp_payload", "$.TouchPoint.Attributes.DTCFlag").alias("tcpm_DTCFlag"),
        F.get_json_object("slntp_payload", "$.TouchPoint.Attributes.Region").alias("tcpm_Region"),
        F.get_json_object("slntp_payload", "$.TouchPoint.Attributes.City").alias("tcpm_City"),
        F.get_json_object("slntp_payload", "$.TouchPoint.Attributes.CustomAttributeList.CustomAttribute").alias("tcpm_Attr_CustomAttributeList"),
        
        # # >>> 新增：存储原始嵌套结构为 JSON 字符串 <<<
        F.get_json_object("slntp_payload", "$.TouchPoint.ContactInformation.PhoneList.Phone").alias("tcpm_PHONELIST"),
        F.get_json_object("slntp_payload", "$.TouchPoint.ContactInformation.AddressList.Address").alias("tcpm_ADDRESSLIST"),
        F.get_json_object("slntp_payload", "$.TouchPoint.CustomAttributeList.CustomAttribute").alias("tcpm_CUSTOMATTRIBUTELIST"),
        F.get_json_object("slntp_payload", "$.TouchPoint.TerminalRegistrationList.TerminalRegistration").alias("tcpm_TERMINALREGISTRATIONLIST"),

        # # Organization Hierarchy
        org_field("Global", "Level").alias("tcpm_GLOBAL_Level"),
        org_field("Global", "Code").alias("tcpm_GLOBAL_CODE"),
        org_field("Global", "Name").alias("tcpm_GLOBAL_NAME"),
        org_field("Global", "Description").alias("tcpm_GLOBAL_DESCRIPTION"),
        org_field("Regional", "Level").alias("tcpm_REGIONAL_Level"),
        org_field("Regional", "Code").alias("tcpm_REGIONAL_CODE"),
        org_field("Regional", "Name").alias("tcpm_REGIONAL_NAME"),
        org_field("Regional", "Description").alias("tcpm_REGIONAL_DESCRIPTION"),
        org_field("Affiliate", "Level").alias("tcpm_AFFILIATE_Level"),
        org_field("Affiliate", "Code").alias("tcpm_AFFILIATE_CODE"),
        org_field("Affiliate", "Name").alias("tcpm_AFFILIATE_NAME"),
        org_field("Affiliate", "Description").alias("tcpm_AFFILIATE_DESCRIPTION"),

        # AuxiliarySourceSystem 重新获取(talend 历史数据存在bug,老数据未正常解析)
        F.get_json_object("slntp_payload", "$.TouchPoint.AuxiliarySourceSystem.@Code").alias("tcpm_auxiliarycode_new"),  
        F.get_json_object("slntp_payload", "$.TouchPoint.AuxiliarySourceSystem.TouchPointCode").alias("tcpm_auxiliarytouchpointcode_new"),

        F.get_json_object("slntp_payload", "$.TouchPoint.SourceSystem.SourceTimestamp").alias("tcpm_sourcetimestamp_new"),
    )

    return json_parse_df

In [0]:
import pyspark.sql.functions as F


def get_target_table_name(tartget_db: str, tartget_tb: str) -> str:
    return f"{tartget_db}.{tartget_tb}"


current_timesamp_str = "20260612120000"

CONFIG_TABLE = "touchpoint_data_loading_delta_config"  
data_loading_config =spark.read.option("header", "true").csv(f"/Volumes/catalog_southeastasia_mdm_pr/share_mdm_config/mdm_config_files/cdp_mdm_pr/touchpoint_retry/pr_touchpoint_data_loading_delta_config_20260618.csv")
data_loading_config.createOrReplaceTempView(f"{CONFIG_TABLE}")

# 读取配置表，只处理激活状态的数据 
config_df = spark.table(CONFIG_TABLE).filter("is_loading_delta_active = true  ").orderBy(col("source_table").asc(),col("market").asc())

landing_json_df_config = spark.table(CONFIG_TABLE).filter("source_table = 'slandtouchpoint' ")
rtouchpointmaster_df_config = spark.table(CONFIG_TABLE).filter("source_table = 'rtouchpointmaster' ")
touchpointmasterlist_df_config = spark.table(CONFIG_TABLE).filter("source_table = 'touchpointmasterlist' ")

i = 1

# 遍历每一条配置（逐表同步）
for row in config_df.collect():
    # 提取配置字段
    market = row["market"].upper()
    source_db = row["source_database"]
    source_tb = row["source_table"]
    tartget_db = row["tartget_database"]
    tartget_tb = row["target_table"]

    target_blob_path = row["target_path"]  # 源 Delta Blob 路径
    primary_key = row["primary_key"].strip().upper()  # 主键（支持单个/多个逗号分隔）
    # pk_type = row["pk_type"]  # 主键类型（本次自动用源表Schema，无需手动指定）
    Markt_Column = row["Markt_Column"]

    # 目标表名
    target_table = get_target_table_name(tartget_db, tartget_tb)

    print("="*150)
    print(i)
    print(f"market: {market}")
    i = i + 1 
    print(f"=== 开始同步：{source_db}.{source_tb} -> {target_table} | 主键：{primary_key},{Markt_Column} ===")


    # step01: 数据生成
    print(f"source table path: {target_blob_path}")
    source_df = spark.read.format("delta").load(target_blob_path)
    
    print("source_df: ")
    source_df.groupBy(Markt_Column).count().show()

    
    if market == "APAC":
        if source_tb == "touchpointsapbi":

            result_df = source_df \
                .withColumn("tp_sapbi_id", F.concat(F.lit("APAC-"), F.col("tp_sapbi_id"))) \
                .withColumn("batch_id", F.concat(F.lit("init-"), F.lit(current_timesamp_str))) \
                .withColumn("creation_dt", F.current_timestamp()) \
                .withColumn("update_dt", F.current_timestamp()) \
                .withColumn("source_type", F.lit("APAC")) \
                .withColumn("file_date", F.regexp_extract(col("filename_sapbi"), r"_(\d{4}-\d{2}-\d{2}).*\.csv$", 1))
        
        elif source_tb == "rtouchpointmaster":
            # 1.补充json字段  
            # todo 补充从json中获取 TCPM_DocumentTimestamp, TCPM_SourceTimestamp, TCPM_OpenDate, TCPM_Active 字段
            apac_landing_json_path = landing_json_df_config.filter("market = 'APAC' ").head()["target_path"]
            print(f"apac_landing_json_path: {apac_landing_json_path}")
            json_df = get_add_json_column(spark.read.format("delta").load(apac_landing_json_path))


            # 2.字段修改
            update_df = source_df.join(json_df, ["tcpm_tcpt_id"], "left") \
                .withColumn("batch_id", F.concat(F.lit("init-"), F.lit(current_timesamp_str))) \
                .withColumn("tcpm_id", F.concat(F.lit("APAC-"), F.col("tcpm_id"))) \
                .withColumn("tcpm_tcpt_id", F.concat(F.lit("APAC-"), F.col("tcpm_tcpt_id"))) \
                .withColumn("tcpm_auxiliarycode", F.col("tcpm_auxiliarycode_new")) \
                .withColumn("tcpm_auxiliarytouchpointcode", F.col("tcpm_auxiliarytouchpointcode_new")) \
                .withColumn("tcpm_sourcetimestamp", F.col("tcpm_sourcetimestamp_new")) \
                .drop("tcpm_auxiliarycode_new", "tcpm_auxiliarytouchpointcode_new", "tcpm_sourcetimestamp_new")

            # 3. 排除数据
            correct_df = update_df.filter(F.coalesce(F.col("TCPM_MarketCode"), F.lit("unknow")) != "KOR")
            print("correct_df: ")
            correct_df.groupBy(Markt_Column).count().show()

            error_df = update_df.filter(F.col("TCPM_MarketCode") == "KOR")
            print("error_df: ")
            error_df.groupBy(Markt_Column).count().show()

            other_market_data_path = rtouchpointmaster_df_config.where("market = 'KOR' ").head()["target_path"]
            print(f"other_market_data_path: {other_market_data_path}")
            other_market_data_df =  spark.read.format("delta").load(other_market_data_path) 

            pass_exclued_df = (error_df.alias("ed")
                          .join(other_market_data_df.alias("omdd"), ["tcpm_marketcode", "tcpm_brandcode", "tcpm_sourcesystemcode", "tcpm_touchpointcode"], "left")
                          .filter(
                            (F.col("omdd.tcpm_marketcode").isNull()) 
                            # | (F.col("cd.TCPM_SourceTimestamp") > F.col("odd.TCPM_SourceTimestamp"))
                          )
                          .select(F.col("ed.*"))
                          .distinct()
                          )
            print("pass_exclued_df: ")
            pass_exclued_df.groupBy(Markt_Column).count().show()
            
            result_df = correct_df.unionByName(pass_exclued_df)
            print("result_df: ")
            result_df.groupBy(Markt_Column).count().show()

        elif source_tb == "touchpointmasterlist":
            
            # 1. 字段更名
            update_df = source_df \
                .withColumn("batch_id", F.concat(F.lit("init-"), F.lit(current_timesamp_str))) \
                .withColumn("tcpt_id", F.concat(F.lit("APAC-"), F.col("slntp_id"))) \
                .drop("slntp_id") \
                # .where("""
                #    !(
                #     (MarketCode = 'KOR' and BrandCode = '03' and TouchPointCode = 'DSW_005') or 
                #     (MarketCode = 'KOR' and BrandCode = '03' and TouchPointCode = 'DSW_011') or 
                #     (MarketCode = 'KOR' and BrandCode = '03' and TouchPointCode = 'DSW_047') or 
                #     (MarketCode = 'KOR' and BrandCode = '07' and TouchPointCode = 'DSW_029') or 
                #     (MarketCode = 'KOR' and BrandCode = '37' and TouchPointCode = 'EC001')
                #     )
                # """) \
            
            # 2. 排除数据
            correct_df = update_df.filter(F.coalesce(F.col("MarketCode"), F.lit("unknow")) != "KOR")
            print("correct_df: ")
            correct_df.groupBy(Markt_Column).count().show()

            error_df = update_df.filter(F.col("MarketCode") == "KOR")
            print("error_df: ")
            error_df.groupBy(Markt_Column).count().show()

            other_market_data_path = touchpointmasterlist_df_config.where("market = 'KOR' ").head()["target_path"]
            print(f"other_market_data_path: {other_market_data_path}")
            other_market_data_df =  spark.read.format("delta").load(other_market_data_path) 

            pass_exclued_df = (error_df.alias("ed")
                          .join(other_market_data_df.alias("omdd"), ["MarketCode", "BrandCode", "TouchPointCode"], "left")
                          .filter(
                            (F.col("omdd.MarketCode").isNull()) 
                            # | (F.col("cd.TCPM_SourceTimestamp") > F.col("odd.TCPM_SourceTimestamp"))
                          )
                          .select(F.col("ed.*"))
                          .distinct()
                          )
            print("pass_exclued_df: ")
            pass_exclued_df.groupBy(Markt_Column).count().show()

            result_df = correct_df.unionByName(pass_exclued_df)
            print("result_df: ")
            result_df.groupBy(Markt_Column).count().show()


        else:
            print("无效table")

    else:
        if source_tb == "touchpointsapbi":

            result_df = source_df \
                .withColumn("tp_sapbi_id", F.concat(F.lit("KOR-"), F.col("tp_sapbi_id"))) \
                .withColumn("batch_id", F.concat(F.lit("init-"), F.lit(current_timesamp_str))) \
                .withColumn("creation_dt", F.current_timestamp()) \
                .withColumn("update_dt", F.current_timestamp()) \
                .withColumn("source_type", F.lit("KOR")) \
                .withColumn("file_date", F.regexp_extract(col("filename_sapbi"), r"_(\d{4}-\d{2}-\d{2}).*\.csv$", 1))

           
        elif source_tb == "rtouchpointmaster":

            # 1.补充json字段
            kor_landing_json_path = landing_json_df_config.filter("market = 'KOR' ").head()["target_path"]
            print(f"kor_landing_json_path; {kor_landing_json_path}")
            json_df = get_add_json_column(spark.read.format("delta").load(kor_landing_json_path))

            # 2.字段修改
            update_df = source_df.join(json_df, ["tcpm_tcpt_id"], "left") \
                .withColumn("batch_id", F.concat(F.lit("init-"), F.lit(current_timesamp_str))) \
                .withColumn("tcpm_id", F.concat(F.lit("KOR-"), F.col("tcpm_id"))) \
                .withColumn("tcpm_tcpt_id", F.concat(F.lit("KOR-"), F.col("tcpm_tcpt_id"))) \
                .withColumn("tcpm_auxiliarycode", F.col("tcpm_auxiliarycode_new")) \
                .withColumn("tcpm_auxiliarytouchpointcode", F.col("tcpm_auxiliarytouchpointcode_new")) \
                .withColumn("tcpm_sourcetimestamp", F.col("tcpm_sourcetimestamp_new")) \
                .drop("tcpm_auxiliarycode_new", "tcpm_auxiliarytouchpointcode_new", "tcpm_sourcetimestamp_new")
            
            # 3. 排除数据
            correct_df = update_df.filter( F.coalesce(F.col("TCPM_MarketCode"), F.lit("unknow")) == "KOR")
            print("correct_df: ")
            correct_df.groupBy(Markt_Column).count().show()

            error_df = update_df.filter(F.coalesce(F.col("TCPM_MarketCode"), F.lit("unknow")) != "KOR")  
            print("error_df: ")
            error_df.groupBy(Markt_Column).count().show()

            other_market_data_path = rtouchpointmaster_df_config.where("market = 'APAC' ").head()["target_path"]
            print(f"other_market_data_path: {other_market_data_path}")
            other_market_data_df =  spark.read.format("delta").load(other_market_data_path) 

            pass_exclued_df = (error_df.alias("ed")
                          .join(other_market_data_df.alias("omdd"), ["tcpm_marketcode", "tcpm_brandcode", "tcpm_sourcesystemcode", "tcpm_touchpointcode"], "left")
                          .filter(
                            (F.col("omdd.tcpm_marketcode").isNull()) 
                            # | (F.col("cd.TCPM_SourceTimestamp") > F.col("odd.TCPM_SourceTimestamp"))
                          )
                          .select(F.col("ed.*"))
                          .distinct()
                          )
            print("pass_exclued_df: ")
            pass_exclued_df.groupBy(Markt_Column).count().show()
            
            result_df = correct_df.unionByName(pass_exclued_df)
            result_df.groupBy(Markt_Column).count().show()
        
        elif source_tb == "touchpointmasterlist":
            
            # 字段更名
            update_df = source_df \
                .withColumn("batch_id", F.concat(F.lit("init-"), F.lit(current_timesamp_str))) \
                .withColumn("tcpt_id", F.concat(F.lit("KOR-"), F.col("slntp_id"))) \
                .drop("slntp_id")

            # 2. 排除数据
            correct_df = update_df.filter(F.coalesce(F.col("MarketCode"), F.lit("unknow")) == "KOR")
            print("correct_df: ")
            correct_df.groupBy(Markt_Column).count().show()

            error_df = update_df.filter(F.coalesce(F.col("MarketCode"), F.lit("unknow")) != "KOR")
            print("error_df: ")
            error_df.groupBy(Markt_Column).count().show()

            other_market_data_path = touchpointmasterlist_df_config.where("market = 'APAC' ").head()["target_path"]
            print(f"other_market_data_path: {other_market_data_path}")
            other_market_data_df =  spark.read.format("delta").load(other_market_data_path) 

            pass_exclued_df = (error_df.alias("ed")
                          .join(other_market_data_df.alias("omdd"), ["MarketCode", "BrandCode", "TouchPointCode"], "left")
                          .filter(
                            (F.col("omdd.MarketCode").isNull()) 
                            # | (F.col("cd.TCPM_SourceTimestamp") > F.col("odd.TCPM_SourceTimestamp"))
                          )
                          .select(F.col("ed.*"))
                          .distinct()
                          )
            print("pass_exclued_df: ")
            pass_exclued_df.groupBy(Markt_Column).count().show()

            result_df = correct_df.unionByName(pass_exclued_df)
            print("result_df: ")
            result_df.groupBy(Markt_Column).count().show()

        else:
            print("!"*100)
            print("无效table")



    # step02: 非market数据额外储存
    # target_blob_path 同层目录下
    # if error_df.count() > 0:
    #     filter_path =  target_blob_path.replace("Touchpint_20260408", "Touchpint_20260408/Filter_Data")  
    #     print(f"filter_path: {filter_path}")
    #     error_df.write \
    #         .format("delta") \
    #         .mode("append") \
    #         .saveAsTable(filter_path)



    target_df = spark.table(target_table)

    compare_schemas(result_df, target_df, source_name="Source(Blob)表", target_name="Target(Databricks)表")
    write_df = align_schema(result_df,target_df)

    

    # 直接append
    (write_df.write.format("delta").mode("append").saveAsTable(target_table))


print("=== 所有配置表任务执行完毕 ===")

In [0]:
"""
0. check apac 哪些code需要排除

1. 执行history loading 脚本
    t_touchpoint_sapbi_source, t_touchpoint_master, t_touchpoint_dataset

2. check 数据

3. 手动执行 t_touchpoint_sapbi,  t_touchpoint_master_sap, t_touchpoint_dataset


"""

#uat history check

##1. touchpointsapbi

In [0]:
%sql

select
  count(*)
from
  delta.`abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_pr/history_data_loading/Pr_Touchpoint_20260610/APAC/touchpointsapbi`


In [0]:
%sql

select
  count(*)
from
  delta.`abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_pr/history_data_loading/Pr_Touchpoint_20260610/KOR/touchpointsapbi`


In [0]:
%sql

select
  -- source_type,
  count(*)
from
  ( 
    select 
      'APAC' as source_type,* 
    from 
      delta.`abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_pr/history_data_loading/Pr_Touchpoint_20260610/APAC/touchpointsapbi`
    union
    select 
      'KOR' as source_type,* 
    from 
      delta.`abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_pr/history_data_loading/Pr_Touchpoint_20260610/KOR/touchpointsapbi`
  )

-- group by
--   source_type

In [0]:
%sql

select
  *
from
  catalog_southeastasia_mdm_pr.touchpoint_parsed.t_touchpoint_sapbi_source

In [0]:
%sql

select
  -- source_type, 
  count(*), min(file_date), max(file_date)
from
  catalog_southeastasia_mdm_pr.touchpoint_parsed.t_touchpoint_sapbi_source
-- group by
--   source_type2

In [0]:
%sql

select
  marketcode, divisioncode, businesstype, customergroup,
  count(*)
from
  ( 
     select 
      'APAC' as source_type,* 
    from 
      delta.`abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_pr/history_data_loading/Pr_Touchpoint_20260610/APAC/touchpointsapbi`
    union
    select 
      'KOR' as source_type,* 
    from 
      delta.`abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_pr/history_data_loading/Pr_Touchpoint_20260610/KOR/touchpointsapbi`
  )

group by
  -- marketcode, divisioncode, businesstype, customergroup
  all
order by
  all

In [0]:
%sql

select
  marketcode, divisioncode, businesstype, customergroup,
  count(*)
from
  catalog_southeastasia_mdm_pr.touchpoint_parsed.t_touchpoint_sapbi_source
group by
  all
order by
  all

In [0]:
%sql

select 
  *
from 
  delta.`abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_pr/history_data_loading/Pr_Touchpoint_20260610/APAC/touchpointsapbi`

In [0]:
%sql

select 
  *
from 
  delta.`abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_pr/history_data_loading/Pr_Touchpoint_20260610/KOR/touchpointsapbi`

In [0]:
%sql

select 
  *
from 
  catalog_southeastasia_mdm_pr.touchpoint_parsed.t_touchpoint_sapbi_source
where
  TP_SAPBI_ID = 'APAC-583474' or TP_SAPBI_ID = 'KOR-351330'

In [0]:
%sql

select
  DivisionCode, CustomerNumber
from
  catalog_southeastasia_mdm_pr.touchpoint_parsed.t_touchpoint_sapbi_source
group by
  DivisionCode, CustomerNumber
having
  count(*) > 1

##2. rtouchpointmaster

In [0]:
%sql

-- 总条数
select
  -- source_type,
  count(*)
from
  ( 
    select 
      'APAC' as source_type,* 
    from 
      delta.`abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_pr/history_data_loading/Pr_Touchpoint_20260610/APAC/rtouchpointmaster`
    union
    select 
      'KOR' as source_type,* 
    from 
      delta.`abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_pr/history_data_loading/Pr_Touchpoint_20260610/KOR/rtouchpointmaster`
  )

-- group by
--   source_type

In [0]:
%sql

-- 重复数据
select
  a.*,b.*
from 
  delta.`abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_pr/history_data_loading/Pr_Touchpoint_20260610/APAC/rtouchpointmaster` as a
inner join
  delta.`abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_pr/history_data_loading/Pr_Touchpoint_20260610/KOR/rtouchpointmaster` as b
on
  a.tcpm_marketcode = b.tcpm_marketcode AND
  a.tcpm_brandcode = b.tcpm_brandcode AND
  a.tcpm_touchpointcode = b.tcpm_touchpointcode AND
  a.tcpm_sourcesystemcode = b.tcpm_sourcesystemcode
  

In [0]:
%sql

select
  *
from
  catalog_southeastasia_mdm_pr.touchpoint_master.t_touchpoint_master


In [0]:
%sql

select
  count(*)
from
  catalog_southeastasia_mdm_pr.touchpoint_master.t_touchpoint_master


In [0]:
%sql

select
  count(distinct TCPM_MarketCode,TCPM_BrandCode, TCPM_TouchPointCode)
from
  catalog_southeastasia_mdm_pr.touchpoint_master.t_touchpoint_master


In [0]:
%sql

select
  *
from
    catalog_southeastasia_mdm_pr.touchpoint_master.t_touchpoint_master
where
  tcpm_MarketCode is  null or   tcpm_BrandCode is  null or  tcpm_TouchPointCode is  null

In [0]:
%sql

select
  TCPM_MarketCode, TCPM_BrandCode, TCPM_TouchPointTypeCode,TCPM_RetailerHierarchyCode, TCPM_Active, count(*)
from

  (select 
    apac_tab.* 
  from 
    delta.`abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_pr/history_data_loading/Pr_Touchpoint_20260610/APAC/rtouchpointmaster` as apac_tab
  left anti join
    -- 重复数据
    (
      select
        distinct a.tcpm_marketcode, a.tcpm_brandcode, a.tcpm_touchpointcode, a.tcpm_sourcesystemcode
      from 
        delta.`abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_pr/history_data_loading/Pr_Touchpoint_20260610/APAC/rtouchpointmaster` as a
      inner join
        delta.`abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_pr/history_data_loading/Pr_Touchpoint_20260610/KOR/rtouchpointmaster` as b
      on
        a.tcpm_marketcode = b.tcpm_marketcode AND
        a.tcpm_brandcode = b.tcpm_brandcode AND
        a.tcpm_touchpointcode = b.tcpm_touchpointcode AND
        a.tcpm_sourcesystemcode = b.tcpm_sourcesystemcode
    ) as cross_tab
  on
    apac_tab.tcpm_marketcode = cross_tab.tcpm_marketcode AND
    apac_tab.tcpm_brandcode = cross_tab.tcpm_brandcode AND
    apac_tab.tcpm_touchpointcode = cross_tab.tcpm_touchpointcode AND
    apac_tab.tcpm_sourcesystemcode = cross_tab.tcpm_sourcesystemcode

  union
    
  select 
    * 
  from 
    delta.`abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_pr/history_data_loading/Pr_Touchpoint_20260610/KOR/rtouchpointmaster`
  )
group by
  all
order by
  all

In [0]:
%sql

select
  TCPM_MarketCode, TCPM_BrandCode, TCPM_TouchPointTypeCode,TCPM_RetailerHierarchyCode, TCPM_Active, count(*)
from
  catalog_southeastasia_mdm_pr.touchpoint_master.t_touchpoint_master
group by
  all
order by
  all

In [0]:
%sql
select
  *
from 
  delta.`abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_pr/history_data_loading/Pr_Touchpoint_20260610/APAC/rtouchpointmaster`

In [0]:
%sql
select
  *
from 
  delta.`abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_pr/history_data_loading/Pr_Touchpoint_20260610/KOR/rtouchpointmaster`

In [0]:
%sql

select
  TCPM_ID,TCPM_TCPT_ID,TCPM_SourceSystemCode,TCPM_SourceTimestamp,TCPM_MarketCode,TCPM_AffiliateCode,TCPM_DivisionCode,TCPM_BrandCode,TCPM_TouchPointCode,TCPM_SubMarketCode,TCPM_AuxiliaryCode,TCPM_AuxiliaryTouchPointCode,TCPM_HygieneServiceCode,TCPM_DistributionChannelCode,TCPM_TouchPointGroupCode,TCPM_RetailerHierarchyCode,TCPM_TouchPointTypeCode,TCPM_EnglishDescription,TCPM_LocalDescription,TCPM_EnglishFullDescription,TCPM_LocalFullDescription,TCPM_Descriptionen,TCPM_Descriptionlocal,TCPM_FullDescriptionen,TCPM_FullDescriptionlocal,TCPM_URL,TCPM_JDECode,TCPM_Active,TCPM_TouchPointStatus,TCPM_OpenDate,TCPM_BranchID,TCPM_CustomerNumber,TCPM_CREATION_DT,TCPM_CREATIONUID,TCPM_UPDATE_DT,TCPM_UPDATEUID,TCPM_RedirectTouchPointCode,TCPM_Channel,TCPM_CustomerGroup,TCPM_DTCFlag,TCPM_Region,TCPM_City,TCPM_Attr_CustomAttributeList,TCPM_PHONELIST,TCPM_ADDRESSLIST,TCPM_CUSTOMATTRIBUTELIST,TCPM_TERMINALREGISTRATIONLIST,TCPM_GLOBAL_Level,TCPM_Global_Code,TCPM_Global_Name,TCPM_GLOBAL_DESCRIPTION,TCPM_REGIONAL_Level,TCPM_Regional_Code,TCPM_Regional_Name,TCPM_REGIONAL_DESCRIPTION,TCPM_AFFILIATE_Level,TCPM_Affiliate_Code,TCPM_Affiliate_Name,TCPM_AFFILIATE_DESCRIPTION,TCPM_Action,TCPM_DocumentTimestamp,TCPM_DocumentUUID,TCPM_RecordUUID,BATCH_ID,KAFKA_TIMESTAMP

from
  catalog_southeastasia_mdm_pr.touchpoint_master.t_touchpoint_master
where
  TCPM_ID = 'APAC-5448' or TCPM_ID = 'KOR-1312'


In [0]:
%sql

select
  *
from
  catalog_southeastasia_mdm_pr.touchpoint_master.t_touchpoint_master_sap


In [0]:
%sql

select
  count(*)
from
  catalog_southeastasia_mdm_pr.touchpoint_master.t_touchpoint_master_sap


In [0]:
%sql

select
  count(*)
from
  catalog_southeastasia_mdm_pr.touchpoint_master.t_touchpoint_master_sap
where
  MarketCode is not null and  BrandCode is not null and  TouchPointCode is not null



In [0]:
%sql

select
  MarketCode, BrandCode, TouchPointCode, count(*)
from
  catalog_southeastasia_mdm_pr.touchpoint_master.t_touchpoint_master_sap
group by
  MarketCode, BrandCode, TouchPointCode
having
  count(*) > 1


##3

In [0]:
%sql
-- 总条数
select
  
  count(*)
from
  ( 
    select 
      *
    from 
      delta.`abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_pr/history_data_loading/Pr_Touchpoint_20260610/APAC/touchpointmasterlist`
    union
    select 
      *
    from 
      delta.`abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_pr/history_data_loading/Pr_Touchpoint_20260610/KOR/touchpointmasterlist`
  )



In [0]:
%sql

-- 重复数据
select
  a.*,b.*
from 
  delta.`abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_pr/history_data_loading/Pr_Touchpoint_20260610/APAC/touchpointmasterlist` as a
inner join
  delta.`abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_pr/history_data_loading/Pr_Touchpoint_20260610/KOR/touchpointmasterlist` as b
on
  a.MarketCode = b.MarketCode AND
  a.BrandCode = b.BrandCode AND
  a.TouchPointCode = b.TouchPointCode 

In [0]:
%sql

select
  *
from
  catalog_southeastasia_mdm_pr.touchpoint_combine.t_touchpoint_dataset


In [0]:
%sql

desc history catalog_southeastasia_mdm_pr.touchpoint_combine.t_touchpoint_dataset

In [0]:
%sql

select
  count(*)
from
  catalog_southeastasia_mdm_pr.touchpoint_combine.t_touchpoint_dataset

In [0]:
%sql

select
    b.*
from

(select
  MarketCode, BrandCode, TouchPointCode
from
  catalog_southeastasia_mdm_pr.touchpoint_combine.t_touchpoint_dataset


except

select
  MarketCode, BrandCode, TouchPointCode
from
  catalog_southeastasia_mdm_pr.touchpoint_combine.t_touchpoint_dataset @v2
)  as a

left join
    catalog_southeastasia_mdm_pr.touchpoint_combine.t_touchpoint_dataset as b
on
    a.MarketCode = b.MarketCode and
    a.BrandCode = b.BrandCode and
    a.TouchPointCode = b.TouchPointCode


In [0]:
%sql

select
  marketcode, brandcode, divisioncode,count(*)
from

  (select 
    apac_tab.* 
  from 
    delta.`abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_pr/history_data_loading/Pr_Touchpoint_20260610/APAC/touchpointmasterlist` as apac_tab
  left anti join
    -- 重复数据
    (
      select
        distinct a.MarketCode, a.BrandCode, a.TouchPointCode
      from 
        delta.`abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_pr/history_data_loading/Pr_Touchpoint_20260610/APAC/touchpointmasterlist` as a
      inner join
        delta.`abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_pr/history_data_loading/Pr_Touchpoint_20260610/KOR/touchpointmasterlist` as b
      on
        a.MarketCode = b.MarketCode AND
        a.BrandCode = b.BrandCode AND
        a.TouchPointCode = b.TouchPointCode 
    ) as cross_tab
  on
    apac_tab.MarketCode = cross_tab.MarketCode AND
    apac_tab.BrandCode = cross_tab.BrandCode AND
    apac_tab.TouchPointCode = cross_tab.touchpointcode

  union
    
  select 
    * 
  from 
    delta.`abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_pr/history_data_loading/Pr_Touchpoint_20260610/KOR/touchpointmasterlist`
  )
group by
  all
order by
  all

In [0]:
%sql

select
  marketcode, brandcode, divisioncode,count(*)
from
  catalog_southeastasia_mdm_pr.touchpoint_combine.t_touchpoint_dataset @v2
group by
  all
order by
  all


In [0]:
%sql
select
  *
from 
  delta.`abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_pr/history_data_loading/Pr_Touchpoint_20260610/APAC/touchpointmasterlist`

In [0]:
%sql
select
  *
from 
  delta.`abfss://bronze@saapseaprodacdgen2.dfs.core.windows.net/cdp_mdm_pr/history_data_loading/Pr_Touchpoint_20260610/KOR/touchpointmasterlist`

In [0]:
%sql

select
  *
from
  catalog_southeastasia_mdm_pr.touchpoint_combine.t_touchpoint_dataset @v2
where
  TCPT_ID in ('APAC-357511', 'KOR-439261')


In [0]:
%sql

desc history catalog_southeastasia_mdm_pr.touchpoint_combine.t_touchpoint_dataset

In [0]:
%sql

select
  *
from
  catalog_southeastasia_mdm_pr.touchpoint_combine.t_touchpoint_dataset @v

In [0]:
%sql

select
  count(*)
from
  catalog_southeastasia_mdm_pr.touchpoint_combine.t_touchpoint_dataset

In [0]:
%sql

select
  *
from
  catalog_southeastasia_mdm_pr.touchpoint_combine.t_touchpoint_dataset

In [0]:
%sql

SELECT * FROM table_changes('catalog_southeastasia_mdm_pr.touchpoint_combine.t_touchpoint_dataset', 6,6) where _change_type != 'update_preimage'

In [0]:
%sql

SELECT _change_type,count(*) FROM table_changes('catalog_southeastasia_mdm_pr.touchpoint_combine.t_touchpoint_dataset', 6,6) GROUP BY _change_type

In [0]:
%sql

SELECT * FROM table_changes('catalog_southeastasia_mdm_pr.touchpoint_combine.t_touchpoint_dataset', 6,6) 
-- where _change_type != 'update_preimage'